# EV Adoption Behavior - 03. Exploratory Data Analysis

Structured around 5 themes so the eventual Streamlit dashboard's sections fall out naturally: **Demographics → Economics/Cost → Infrastructure/Range → Psychology/Awareness → Synthesis**.

Reads from `ev_adoption_cleaned.csv` (output of `02_data_cleaning.ipynb`) - never the raw file.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

df = pd.read_csv('ev_adoption_cleaned.csv')

# Fix category ORDER (not values) so every chart shows Low -> Medium -> High
# consistently, instead of pandas defaulting to alphabetical (High, Low, Medium).
adoption_order = ['Low', 'Medium', 'High']
df['ev_adoption_likelihood'] = pd.Categorical(
    df['ev_adoption_likelihood'], categories=adoption_order, ordered=True
)
df.shape

## Recap: target class balance

Keep this visible throughout - it's the lens every other chart gets read through.

In [ ]:
balance = df['ev_adoption_likelihood'].value_counts(normalize=True).reindex(adoption_order) * 100
balance.plot(kind='bar', color=['#e74c3c', '#f39c12', '#2ecc71'])
plt.title('Target Class Balance: ev_adoption_likelihood')
plt.ylabel('% of customers')
plt.xticks(rotation=0)
plt.show()
balance

---
## Section A - Demographic Profile

Age, income, and city_type together, rather than as isolated charts - so you can compare which one actually separates the classes most, instead of assuming each matters equally.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='ev_adoption_likelihood', y='age', order=adoption_order, ax=axes[0])
axes[0].set_title('Age by Adoption Likelihood')

sns.boxplot(data=df, x='ev_adoption_likelihood', y='annual_income', order=adoption_order, ax=axes[1])
axes[1].set_title('Income by Adoption Likelihood')

city_props = pd.crosstab(df['city_type'], df['ev_adoption_likelihood'], normalize='index')[adoption_order] * 100
city_props.plot(kind='bar', stacked=True, ax=axes[2], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[2].set_title('Adoption Likelihood % by City Type')
axes[2].legend(title='Likelihood', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

**Read this as:** boxplots show the *spread* of age/income within each adoption group (overlapping boxes = weak separation, distinct boxes = strong separation). The stacked bar shows what % of each city type falls into each likelihood band - compare bar heights across city types, not raw counts.

---
## Section B - Economics: Cost Comparison

A direct fuel-vs-charging cost comparison - the most "pitch-able" chart for a Marketing audience: concrete monthly savings, not just an abstract score.

In [ ]:
cost_by_group = df.groupby('ev_adoption_likelihood', observed=True)[
    ['fuel_expense_per_month', 'monthly_charging_cost']
].mean().reindex(adoption_order)

cost_by_group.plot(kind='bar', color=['#3498db', '#2ecc71'])
plt.title('Avg Monthly Fuel vs. Charging Cost, by Adoption Likelihood')
plt.ylabel('₹ per month')
plt.xticks(rotation=0)
plt.show()

cost_by_group['potential_monthly_savings'] = (
    cost_by_group['fuel_expense_per_month'] - cost_by_group['monthly_charging_cost']
)
cost_by_group

---
## Section C - Infrastructure & Range

Home charging availability, actual distance to nearest station (objective), and range anxiety (subjective/perceived) - checked separately since they don't necessarily tell the same story.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

home_charge_props = pd.crosstab(
    df['home_charging_available'], df['ev_adoption_likelihood'], normalize='index'
)[adoption_order] * 100
home_charge_props.plot(kind='bar', stacked=True, ax=axes[0], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[0].set_title('Adoption % by Home Charging Availability')
axes[0].set_xticklabels(['No', 'Yes'], rotation=0)

sns.boxplot(data=df, x='ev_adoption_likelihood', y='nearest_charging_station_km', order=adoption_order, ax=axes[1])
axes[1].set_title('Distance to Nearest Station (actual km)')

sns.boxplot(data=df, x='ev_adoption_likelihood', y='range_anxiety_score', order=adoption_order, ax=axes[2])
axes[2].set_title('Range Anxiety Score (perceived)')

plt.tight_layout()
plt.show()

**Worth noticing:** if the objective distance chart and the perceived range-anxiety chart tell *different* stories (e.g., distance barely matters but anxiety strongly does), that itself is a finding - it suggests the barrier is psychological/perception, not actually physical infrastructure. Worth calling out explicitly if you see this.

---
## Section D - Psychology & Awareness

You have 6 "score" columns that all sound plausible individually (environmental_awareness, tech_affinity, range_anxiety, battery_replacement_concern, ev_knowledge, government_incentive_awareness). Ranking them side-by-side shows which ones actually separate the classes most, instead of treating all 6 as equally important.

In [ ]:
# convert target to numeric (Low=0, Medium=1, High=2) ONLY for correlation ranking
# purposes -- not used anywhere else, since it's an ordinal proxy, not a true number.
target_numeric = df['ev_adoption_likelihood'].cat.codes

score_cols = [
    'environmental_awareness_score', 'technology_affinity_score', 'range_anxiety_score',
    'battery_replacement_concern', 'ev_knowledge_score', 'government_incentive_awareness'
]

correlations = df[score_cols].corrwith(target_numeric).sort_values()
correlations.plot(kind='barh', color='#8e44ad')
plt.title('Correlation of Readiness Scores with Adoption Likelihood')
plt.xlabel('Correlation coefficient')
plt.show()
correlations

In [ ]:
# previous EV experience -- repeat-customer effect
exp_props = pd.crosstab(
    df['previous_ev_experience'], df['ev_adoption_likelihood'], normalize='index'
)[adoption_order] * 100
exp_props.plot(kind='bar', stacked=True, color=['#e74c3c', '#f39c12', '#2ecc71'])
plt.title('Adoption % by Previous EV Experience')
plt.xticks([0, 1], ['No prior experience', 'Prior experience'], rotation=0)
plt.legend(title='Likelihood', bbox_to_anchor=(1.05, 1))
plt.show()

---
## Section E - Synthesis: Best-Fit Customer Segment

Ties the sections above together: which combination of city_type + current_vehicle_type + home_charging_available has the highest *proportion* of High-likelihood customers, and is large enough to be a meaningful segment (not just 3 lucky rows)?

In [ ]:
segment_summary = (
    df.groupby(['city_type', 'current_vehicle_type', 'home_charging_available'], observed=True)
      .agg(
          segment_size=('ev_adoption_likelihood', 'size'),
          pct_high=('ev_adoption_likelihood', lambda x: (x == 'High').mean() * 100)
      )
      .reset_index()
)

# filter out tiny segments that aren't statistically meaningful
segment_summary = segment_summary[segment_summary['segment_size'] >= 200]

top_segments = segment_summary.sort_values('pct_high', ascending=False).head(10)
top_segments

In [ ]:
plt.figure(figsize=(10, 6))
labels = (
    top_segments['city_type'] + ' / ' +
    top_segments['current_vehicle_type'] + ' / ' +
    top_segments['home_charging_available'].map({0: 'no home chg', 1: 'home chg'})
)
plt.barh(labels, top_segments['pct_high'], color='#2ecc71')
plt.xlabel('% of segment classified High adoption likelihood')
plt.title('Top 10 Segments by Adoption Likelihood (min. 200 customers)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Summary - write this up in your own words before moving to `app.py`

For each section above, note 1-2 sentences on what you actually found (not what you expected to find) - this becomes your dashboard's narrative, and the talking points you'll want ready if this comes up in an interview:

- Demographics: 
- Economics: 
- Infrastructure & Range: 
- Psychology & Awareness: 
- Best-fit segment (synthesis): 

## Section A - Demographics
* Age: flat across all three groups (~45 years, Low/Medium/High). Age doesn't separate adopters from non-adopters at all - worth dropping as a targeting variable.
* Income: real gradient - ₹29.4k (Low) → ₹35k (Medium) → ₹52.2k (High). Income clearly matters.
* City type: Urban stands out - 65% High-likelihood vs. ~55% for Rural/Suburban (which are nearly identical to each other). So it's really an Urban vs. everyone-else split, not a 3-way spread.

## Section B - Economics
* Fuel expense (~₹295/month) and charging cost (~₹40/month) are essentially flat across all three adoption groups - Low, Medium, and High spenders all pay about the same for fuel and would save about the same (~₹255/month) by switching. This is a real, slightly counter-intuitive finding: cost savings don't predict who adopts - everyone has similar economics, so cost alone isn't what's separating adopters from non-adopters. Still a useful number for a pitch ("switching saves ~₹255/month"), just not a targeting lever.

## Section C - Infrastructure & Range
* Home charging: 62.5% High-likelihood with home charging available vs. 53.6% without - a meaningful gap.
* Distance to nearest station: mild gradient (8.1km Low → 6.7km High) - matters, but modestly.
* Range anxiety (perceived): sharp gradient - 7.76 (Low) → 6.43 (Medium) → 4.14 (High). This is a much bigger gap than the physical distance numbers show, confirming the Section D finding below: perception matters more than actual proximity.

## Section D - Psychology & Awareness

### Ranked by correlation strength with adoption:

* range_anxiety_score: -0.70 (strongest predictor overall, and negative - lower anxiety = higher adoption)
* ev_knowledge_score: +0.72
* environmental_awareness_score: +0.69
* technology_affinity_score: +0.68
* battery_replacement_concern: -0.15 (weak)
* government_incentive_awareness: +0.12 (weakest - incentive campaigns aren't moving the needle much in this data)
* Previous EV experience: 66.6% High vs. 57.5% for first-timers - a real but moderate repeat-customer effect.

## Section E - Synthesis

Top segments are dominated by Urban + home charging combinations. The largest robust segment (not just a lucky small group) is Urban / Sedan / home charging available - 5,132 customers, 67.9% High-likelihood. Urban/Truck/home-charging scores slightly higher (68.9%) but on a much smaller base (1,490), so it's a less reliable target.

## Limitations
>"Some findings (weak effect of cost savings and incentive awareness) diverge from published EV adoption research, suggesting this dataset may not fully capture real-world causal drivers - findings here reflect patterns in this dataset, not a claim about the broader EV market."

## **Bottom line for your dashboard's closing insight**:

>the strongest actionable profile is urban, higher-income, home-charging-available customers with low range anxiety and above-average EV knowledge - and notably, cost savings and government incentive awareness are not what's driving this, which is a genuinely interesting (and slightly counter-narrative) finding worth highlighting rather than burying.